In [18]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import optuna
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from aml.data.load import load_raw_data

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
import lightgbm as lgb


SEED = 42
TRAIN_SHARE = 0.70
VALIDATION_SHARE = 0.15
NEGATIVE_SAMPLE_RATIO = 0.39
SAVE_SPLITS = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "processed"
SPLIT_OUTPUT_DIR = DATA_DIR

## Loading and validation of raw data

In [3]:
df = load_raw_data()

required_columns = {
    "Timestamp",
    "From Bank",
    "Account",
    "To Bank",
    "Account.1",
    "Amount Received",
    "Receiving Currency",
    "Amount Paid",
    "Payment Currency",
    "Payment Format",
    "Is Laundering",
}
missing_columns = required_columns - set(df.columns)

assert not missing_columns, f"Required columns not found: {sorted(missing_columns)}"
assert df["Timestamp"].is_monotonic_increasing, "Transactions are not sorted by time"
assert set(pd.unique(df["Is Laundering"])).issubset({0, 1}), "The target must be binary"

pd.Series(
    {
        "rows": len(df),
        "columns": df.shape[1],
        "positives": int(df["Is Laundering"].sum()),
        "prevalence": df["Is Laundering"].mean(),
        "start": df["Timestamp"].min(),
        "end": df["Timestamp"].max(),
    },
    name="raw data",
)

rows                      6924041
columns                        15
positives                    3565
prevalence               0.000515
start         2022-09-01 00:00:00
end           2022-09-17 15:28:00
Name: raw data, dtype: object

## Feature Engineering

This part uses only information available at the time of the current transaction:

- matching currencies, banks, and accounts;
- cyclical representation of the hour and day of the week;
- logarithms of amounts, and the discrepancy between the sent and received amounts;
- overnight and weekend transactions;
- indicators of round values.

In [4]:
X = df.drop(columns=["Timestamp", "Is Laundering"]).copy()
timestamp = df["Timestamp"]

# Time
hour = timestamp.dt.hour
day_of_week = timestamp.dt.dayofweek
paid = df["Amount Paid"].clip(lower=0)
received = df["Amount Received"].clip(lower=0)

X["hour"] = hour.astype("int8")
X["dayofweek"] = day_of_week.astype("int8")
X["day"] = timestamp.dt.day.astype("int8")
X["month"] = timestamp.dt.month.astype("int8")

X["hour_sin"] = np.sin(2 * np.pi * hour / 24).astype("float32")
X["hour_cos"] = np.cos(2 * np.pi * hour / 24).astype("float32")
X["dow_sin"] = np.sin(2 * np.pi * day_of_week / 7).astype("float32")
X["dow_cos"] = np.cos(2 * np.pi * day_of_week / 7).astype("float32")
X["is_night"] = hour.between(0, 5).astype("int8")
X["is_weekend"] = day_of_week.isin([5, 6]).astype("int8")

# Type of money
X["is_currency_same"] = (
    df["Receiving Currency"] == df["Payment Currency"]
).astype("int8")
X["is_same_bank"] = (df["From Bank"] == df["To Bank"]).astype("int8")
X["is_self_transfer"] = (df["Account"] == df["Account.1"]).astype("int8")

X["log_amount_paid"] = np.log1p(paid).astype("float32")
X["log_amount_received"] = np.log1p(received).astype("float32")
X["amount_log_gap"] = np.abs(
    X["log_amount_paid"] - X["log_amount_received"]
).astype("float32")
X["is_round_10"] = np.isclose(np.mod(paid, 10), 0, atol=1e-8).astype("int8")
X["is_round_100"] = np.isclose(np.mod(paid, 100), 0, atol=1e-8).astype("int8")
X["is_round_1000"] = np.isclose(np.mod(paid, 1_000), 0, atol=1e-8).astype("int8")

Historical features are calculated across the entire chronological sequence **prior to the temporal split**. This is valid: validation and test rows see only preceding events, just as in production.

`cumcount`, `cumsum - current`, and `diff` are used to exclude the current operation. **The target variable is not included in the calculations**.

In [ ]:
sender = X["Account"]
receiver = X["Account.1"]
paid = X["Amount Paid"].clip(lower=0)

# Cumulative transaction counters
sender_previous_count = X.groupby(sender, sort=False).cumcount().astype("int32")
receiver_previous_count = X.groupby(receiver, sort=False).cumcount().astype("int32")
pair_previous_count = X.groupby([sender, receiver], sort=False).cumcount().astype("int32")

X["sender_prev_tx_log"] = np.log1p(sender_previous_count).astype("float32")
X["receiver_prev_tx_log"] = np.log1p(receiver_previous_count).astype("float32")
X["pair_prev_tx_log"] = np.log1p(pair_previous_count).astype("float32")

# First interaction flag and number of unique counterparties prior to the current transaction
is_new_pair = (pair_previous_count == 0).astype("int32")
X["is_new_pair"] = is_new_pair.astype("int8")

sender_unique_receivers_before = (
    is_new_pair.groupby(sender, sort=False).cumsum() - is_new_pair
)
receiver_unique_senders_before = (
    is_new_pair.groupby(receiver, sort=False).cumsum() - is_new_pair
)
X["sender_unique_receivers_log"] = np.log1p(
    sender_unique_receivers_before
).astype("float32")
X["receiver_unique_senders_log"] = np.log1p(
    receiver_unique_senders_before
).astype("float32")

# Share of the sender's interbank transfers in the past
is_interbank = (X["From Bank"] != X["To Bank"]).astype("int32")
sender_prior_interbank_count = (
    is_interbank.groupby(sender, sort=False).cumsum() - is_interbank
)
X["sender_prior_interbank_ratio"] = (
    sender_prior_interbank_count / sender_previous_count.replace(0, np.nan)
).fillna(0).astype("float32")

# Time (in minutes) since the sender's and recipient's last transaction
sender_minutes_since_previous = (
    df.groupby(sender, sort=False)["Timestamp"].diff().dt.total_seconds() / 60
)
receiver_minutes_since_previous = (
    df.groupby(receiver, sort=False)["Timestamp"].diff().dt.total_seconds() / 60
)
X["sender_minutes_since_prev_log"] = np.log1p(
    sender_minutes_since_previous.clip(lower=0)
).fillna(-1).astype("float32")
X["receiver_minutes_since_prev_log"] = np.log1p(
    receiver_minutes_since_previous.clip(lower=0)
).fillna(-1).astype("float32")

# Average historical interval between sender transactions
sender_interval = sender_minutes_since_previous.fillna(0)
sender_prior_interval_sum = (
    sender_interval.groupby(sender, sort=False).cumsum() - sender_interval
)
sender_prior_average_minutes = (
    sender_prior_interval_sum / sender_previous_count.replace(0, np.nan)
)
X["sender_prior_avg_minutes_log"] = np.log1p(
    sender_prior_average_minutes.clip(lower=0)
).fillna(-1).astype("float32")

# Amount anomaly: the ratio of the current payment to the sender's average historical amount.
sender_prior_amount_sum = paid.groupby(sender, sort=False).cumsum() - paid
sender_prior_amount_mean = (
    sender_prior_amount_sum / sender_previous_count.replace(0, np.nan)
)
amount_to_prior_mean = paid / sender_prior_amount_mean.replace(0, np.nan)
X["amount_to_sender_prior_mean_log"] = (
    np.log1p(amount_to_prior_mean.clip(lower=0, upper=1e6))
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .astype("float32")
)

X.shape

(6924041, 39)

## Timesplit

In [7]:
event_time = df["Timestamp"]
y = df["Is Laundering"]

train_cutoff = event_time.quantile(TRAIN_SHARE)
validation_cutoff = event_time.quantile(TRAIN_SHARE + VALIDATION_SHARE)

train_mask = event_time < train_cutoff
validation_mask = (event_time >= train_cutoff) & (event_time < validation_cutoff)
test_mask = event_time >= validation_cutoff

assert (train_mask.astype("int8") + validation_mask.astype("int8") + test_mask.astype("int8")).eq(1).all()

X_train = X.loc[train_mask].reset_index(drop=True)
y_train = y.loc[train_mask].reset_index(drop=True)
X_val = X.loc[validation_mask].reset_index(drop=True)
y_val = y.loc[validation_mask].reset_index(drop=True)
X_test = X.loc[test_mask].reset_index(drop=True)
y_test = y.loc[test_mask].reset_index(drop=True)

split_summary = pd.DataFrame(
    {
        "rows": [len(y_train), len(y_val), len(y_test)],
        "positives": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
        "prevalence": [y_train.mean(), y_val.mean(), y_test.mean()],
        "start": [
            event_time.loc[train_mask].min(),
            event_time.loc[validation_mask].min(),
            event_time.loc[test_mask].min(),
        ],
        "end": [
            event_time.loc[train_mask].max(),
            event_time.loc[validation_mask].max(),
            event_time.loc[test_mask].max(),
        ],
    },
    index=["train", "validation", "test"],
)
split_summary.style.format({"prevalence": "{:.4%}"})

del X, df, y, event_time, train_mask, validation_mask, test_mask

## Weighted undersampling

In [44]:
y_array = y_train.to_numpy()
positive_positions = np.flatnonzero(y_array == 1)
negative_positions = np.flatnonzero(y_array == 0)

rng = np.random.default_rng(SEED)
negative_sample_size = max(1, int(NEGATIVE_SAMPLE_RATIO * len(negative_positions)))
sampled_negative_positions = rng.choice(
    negative_positions,
    size=negative_sample_size,
    replace=False,
)

fit_positions = np.concatenate([positive_positions, sampled_negative_positions])
rng.shuffle(fit_positions)

X_fit = X_train.iloc[fit_positions].reset_index(drop=True)
y_fit = y_train.iloc[fit_positions].reset_index(drop=True)
negative_weight = len(negative_positions) / negative_sample_size
sample_weight = np.where(y_fit.to_numpy() == 0, negative_weight, 1.0)

## Estimation function

In [12]:
def ranking_metrics(y_true: pd.Series | np.ndarray, scores: np.ndarray) -> dict[str, float]:
    return {
        "PR-AUC": average_precision_score(y_true, scores),
        "ROC-AUC": roc_auc_score(y_true, scores),
    }

## Training

### 1. Logistic Regression

Logistic regression was chosen as the basis due to its simplicity of training and speed of operation.

In [13]:
identifier_columns = ["Account", "Account.1", "From Bank", "To Bank"]
categorical_features = [
    "Payment Format",
    "Receiving Currency",
    "Payment Currency",
]
numeric_features = list(X_fit.select_dtypes(include=np.number).columns)

missing_columns = (
    set(identifier_columns + categorical_features + numeric_features) - set(X_fit.columns)
)
assert not missing_columns, f"Expected signs not found: {sorted(missing_columns)}"

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), categorical_features,
        ),
    ],
    remainder="drop",
    sparse_threshold=1.0,
)

X_fit_lr = preprocessor.fit_transform(X_fit)
X_val_lr = preprocessor.transform(X_val)

logistic_model = LogisticRegression(
    C=1.0,
    penalty="l2",
    random_state=SEED,
)
logistic_model.fit(X_fit_lr, y_fit, sample_weight=sample_weight)

validation_scores = {
    "Logistic Regression": logistic_model.predict_proba(X_val_lr)[:, 1]
}
ranking_metrics(y_val, validation_scores["Logistic Regression"])

{'PR-AUC': 0.08805754770296108, 'ROC-AUC': 0.9599768008999592}

### 2. CatBoost

CatBoost performs well with raw categorical features, including account and bank identifiers, and does not require extensive fine-tuning.

In [14]:
catboost_features = list(X_fit.select_dtypes(include=["object", "string", "category"]).columns)

catboost_model = CatBoostClassifier(
    learning_rate=0.05,
    depth=8,
    l2_leaf_reg=5.0,
    loss_function="Logloss",
    eval_metric="PRAUC:type=Classic",
    random_seed=SEED,
    allow_writing_files=False,
    verbose=100,
)
catboost_model.fit(
    X_fit,
    y_fit,
    cat_features=catboost_features,
    eval_set=(X_val, y_val),
    sample_weight=sample_weight,
    early_stopping_rounds=100,
)

validation_scores["CatBoost"] = catboost_model.predict_proba(X_val)[:, 1]
ranking_metrics(y_val, validation_scores["CatBoost"])

0:	learn: 0.0004553	test: 0.0002807	best: 0.0002807 (0)	total: 729ms	remaining: 12m 8s
100:	learn: 0.2341588	test: 0.2265229	best: 0.2265229 (100)	total: 1m 36s	remaining: 14m 18s
200:	learn: 0.2668366	test: 0.2499612	best: 0.2501689 (199)	total: 3m 44s	remaining: 14m 51s
300:	learn: 0.2817004	test: 0.2516601	best: 0.2516920 (293)	total: 6m 13s	remaining: 14m 28s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 0.2516920282
bestIteration = 293

Shrink model to first 294 iterations.


{'PR-AUC': 0.2518723544435428, 'ROC-AUC': 0.9742155174452463}

### 3. LightGBM

In [25]:
def objective_lgb(trial):
    params = {
        "n_estimators": 2500,
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.12, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 10, 100, log=True),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 300, log=True),
        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 20, log=True),
        "objective": "binary",
        "verbosity": -1,
        "n_jobs": -1,
        "random_state": SEED + trial.number,
    }
    positive_boost = trial.suggest_float("positive_boost", 1.0, 30.0, log=True)
    weights = sample_weight.copy()
    weights[y_fit == 1] *= positive_boost
    model = LGBMClassifier(**params)
    model.fit(X_fit_lr, y_fit, sample_weight=weights,
              eval_set=[(X_val_lr, y_val)],
              eval_metric="average_precision",
              callbacks=[lgb.early_stopping(100, verbose=False)])
    
    score = model.predict_proba(X_val_lr)[:, 1]
    return average_precision_score(y_val, score)

lgb_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
lgb_study.optimize(objective_lgb, n_trials=30, show_progress_bar=True)

print(f"Best Score: {lgb_study.best_value}")
print(f"Best model: {lgb_study.best_params}")

[I 2026-08-15 18:47:30,831] Trial 5 finished with value: 0.26834560252603645 and parameters: {'learning_rate': 0.05948514166655216, 'num_leaves': 20, 'max_depth': 8, 'min_child_samples': 87, 'subsample': 0.7146990594339345, 'colsample_bytree': 0.9893546197175955, 'reg_alpha': 0.7510418138777539, 'reg_lambda': 10.985330528057919, 'positive_boost': 20.978213384576815}. Best is trial 4 with value: 0.2828794536646216.


/Users/artem/Documents/Projects/AML/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[I 2026-08-15 18:47:50,596] Trial 6 finished with value: 0.1733126482101447 and parameters: {'learning_rate': 0.05200543616809575, 'num_leaves': 84, 'max_depth': 4, 'min_child_samples': 33, 'subsample': 0.6658295511186884, 'colsample_bytree': 0.7638656157671425, 'reg_alpha': 0.00877781550471966, 'reg_lambda': 0.014691979880907794, 'positive_boost': 16.755052359850303}. Best is trial 4 with value: 0.2828794536646216.


/Users/artem/Documents/Projects/AML/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[I 2026-08-15 18:48:20,404] Trial 7 finished with value: 0.2689144291792227 and parameters: {'learning_rate': 0.03149717592888798, 'num_leaves': 18, 'max_depth': 8, 'min_child_samples': 29, 'subsample': 0.9307689432639139, 'colsample_bytree': 0.6760927252879199, 'reg_alpha': 8.59873733921227, 'reg_lambda': 2.096273367245241, 'positive_boost': 1.9657448966046123}. Best is trial 4 with value: 0.2828794536646216.


...

[I 2026-08-15 18:59:29,091] Trial 27 finished with value: 0.10596103889621164 and parameters: {'learning_rate': 0.02192199873195868, 'num_leaves': 48, 'max_depth': 9, 'min_child_samples': 103, 'subsample': 0.7410495045438016, 'colsample_bytree': 0.7456104591031084, 'reg_alpha': 0.0002469797143126387, 'reg_lambda': 0.17847227186807574, 'positive_boost': 2.6940153225628114}. Best is trial 11 with value: 0.2888949126786227.


/Users/artem/Documents/Projects/AML/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[I 2026-08-15 18:59:51,708] Trial 28 finished with value: 0.22273870375268795 and parameters: {'learning_rate': 0.017191596938412018, 'num_leaves': 12, 'max_depth': 10, 'min_child_samples': 51, 'subsample': 0.6871885761243751, 'colsample_bytree': 0.8884119966023201, 'reg_alpha': 0.016425349302973097, 'reg_lambda': 0.009638605167867093, 'positive_boost': 1.4889102079780498}. Best is trial 11 with value: 0.2888949126786227.


/Users/artem/Documents/Projects/AML/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[I 2026-08-15 19:00:32,527] Trial 29 finished with value: 0.28265175179913055 and parameters: {'learning_rate': 0.02785055209605628, 'num_leaves': 95, 'max_depth': 10, 'min_child_samples': 89, 'subsample': 0.6947923744272556, 'colsample_bytree': 0.9391127725513246, 'reg_alpha': 0.00016755683778937107, 'reg_lambda': 4.70186033140643, 'positive_boost': 7.735439680560896}. Best is trial 11 with value: 0.2888949126786227.
Best Score: 0.2888949126786227
Best model: {'learning_rate': 0.015135204401864532, 'num_leaves': 52, 'max_depth': 11, 'min_child_samples': 63, 'subsample': 0.9909249788962231, 'colsample_bytree': 0.8304490035666355, 'reg_alpha': 0.002226619249351213, 'reg_lambda': 0.0013268287465198212, 'positive_boost': 29.746908144721015}


In [46]:
lgb_model = LGBMClassifier(**lgb_study.best_params,
                           n_estimators = 2500,
                           objective= "binary",
                           verbosity= -1,
                           n_jobs=-1,)

weights = sample_weight.copy()
weights[y_fit == 1] *= lgb_study.best_params.get('positive_boost')
lgb_model.fit(X_fit_lr, y_fit,
              eval_set=[(X_val_lr, y_val)], 
              sample_weight=weights,
              eval_metric="average_precision",
              callbacks=[lgb.early_stopping(100, verbose=False)])

validation_scores["LGB"] = lgb_model.predict_proba(X_val_lr)[:, 1]
ranking_metrics(y_val, validation_scores["LGB"])

/Users/artem/Documents/Projects/AML/.venv/lib/python3.13/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


{'PR-AUC': 0.18515389905125326, 'ROC-AUC': 0.9725627412187398}

### 4. XGBoost

In [60]:
def objective_xgb(trial):
    positive_boost = trial.suggest_float("positive_boost", 1.0, 30.0, log=True)
    model = XGBClassifier(
        n_estimators=2500,
        learning_rate=trial.suggest_float("learning_rate", 0.015, 0.12, log=True),
        max_depth=trial.suggest_int("max_depth", 3, 8),
        min_child_weight=trial.suggest_float("min_child_weight", 1, 100, log=True),
        subsample=trial.suggest_float("subsample", 0.65, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.65, 1.0),
        gamma=trial.suggest_float("gamma", 1e-5, 5, log=True),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-5, 10, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 20, log=True),
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        max_bin= 128,
        early_stopping_rounds=100,
        n_jobs=-1,
        scale_pos_weight = positive_boost,
        random_state=SEED + trial.number,
    )
    model.fit(
        X_fit_lr, y_fit, sample_weight=sample_weight,
        eval_set=[(X_val_lr, y_val)], verbose=False,
    )
    score = model.predict_proba(X_val_lr)[:, 1]
    return average_precision_score(y_val, score)

xgb_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(objective_xgb, n_trials=20, show_progress_bar=True)

print(f"Best Score: {xgb_study.best_value}")
print(f"Best model: {xgb_study.best_params}")

[I 2026-08-15 19:44:25,669] A new study created in memory with name: no-name-975222ed-f907-4977-bf7c-b2edf290c37d


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-08-15 19:45:14,218] Trial 0 finished with value: 0.2866600714499017 and parameters: {'positive_boost': 3.5747129226002423, 'learning_rate': 0.10831081647339098, 'max_depth': 7, 'min_child_weight': 15.75132049977973, 'subsample': 0.7046065241548528, 'colsample_bytree': 0.7045980821176709, 'gamma': 2.1429733169271365e-05, 'reg_alpha': 1.5741890047456624, 'reg_lambda': 0.3849583075868116}. Best is trial 0 with value: 0.2866600714499017.
[I 2026-08-15 19:47:56,955] Trial 1 finished with value: 0.29297685193625517 and parameters: {'positive_boost': 11.114989443094977, 'learning_rate': 0.015656003500080774, 'max_depth': 8, 'min_child_weight': 46.22589001020831, 'subsample': 0.7243186887373967, 'colsample_bytree': 0.7136387385224853, 'gamma': 0.00011097286548581113, 'reg_alpha': 0.0006690421166498799, 'reg_lambda': 0.1807145634139558}. Best is trial 1 with value: 0.29297685193625517.
[I 2026-08-15 19:50:40,392] Trial 2 finished with value: 0.2794029611305324 and parameters: {'positive

In [63]:
best_params = xgb_study.best_params.copy()
scale_weight = best_params.pop("positive_boost")
best_params["scale_pos_weight"] = scale_weight

xgb_model = XGBClassifier(
    **best_params,
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    early_stopping_rounds=50,
    n_jobs=-1,
    random_state=SEED
)

xgb_model.fit(
    X_fit_lr, 
    y_fit, 
    sample_weight=sample_weight,
    eval_set=[(X_val_lr, y_val)], 
    verbose=False
)

validation_scores["XGB"] = xgb_model.predict_proba(X_val_lr)[:, 1]
print(ranking_metrics(y_val, validation_scores["XGB"]))

{'PR-AUC': 0.26633850168117684, 'ROC-AUC': 0.9713256701290179}


### 5. Extra trees

In [55]:
extra_model = ExtraTreesClassifier(
    n_estimators=100,
    min_samples_leaf=20,
    max_features="sqrt",
    criterion="gini",
    n_jobs=-1,
    random_state=42
)
extra_model.fit(X_fit_lr, y_fit, sample_weight=sample_weight)

validation_scores["Extra"] = extra_model.predict_proba(X_val_lr)[:, 1]
print(ranking_metrics(y_val, validation_scores["Extra"]))

{'PR-AUC': 0.19273893403407216, 'ROC-AUC': 0.97369651405254}


## Model selection on the validation set

In [64]:
validation_results = pd.DataFrame(
    {
        model_name: ranking_metrics(y_val, scores)
        for model_name, scores in validation_scores.items()
    }
).T.sort_values("PR-AUC", ascending=False)

champion_name = validation_results.index[0]
print(f"Champion by validation PR-AUC: {champion_name}")
validation_results.style.format("{:.4f}")

Champion by validation PR-AUC: XGB


,PR-AUC,ROC-AUC
XGB,0.2663,0.9713
CatBoost,0.2519,0.9742
Extra,0.1927,0.9737
Logistic Regression,0.0881,0.9600
LGB,0.0830,0.8533


## Save dataset

In [ ]:
def save_dataset_split(
    X_split: pd.DataFrame,
    y_split: pd.Series,
    output_path: Path,
) -> Path:
    if len(X_split) != len(y_split):
        raise ValueError("The number of rows X and y do not match")

    dataset = X_split.reset_index(drop=True).copy()
    dataset["Is Laundering"] = y_split.reset_index(drop=True).astype("int8")
    dataset.to_parquet(output_path, index=False)
    return output_path


if SAVE_SPLITS:
    SPLIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    saved_paths = {
        "train": save_dataset_split(X_train, y_train, SPLIT_OUTPUT_DIR / "train.parquet"),
        "val": save_dataset_split(X_val, y_val, SPLIT_OUTPUT_DIR / "val.parquet"),
        "test": save_dataset_split(X_test, y_test, SPLIT_OUTPUT_DIR / "test.parquet"),
    }
    display(pd.Series(saved_paths, name="saved to"))
else:
    print("Saving is disabled. Set SAVE_SPLITS = True and execute this cell")